# CatBoost Experiments


## Setup


In [1]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not find repo root (pyproject.toml). Open this notebook from the repo.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
REPO_ROOT


WindowsPath('C:/Users/baben_bakg1j1/HSE/annual_project/stocks-advisor')

In [2]:
import json
import tempfile
from copy import deepcopy
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

from jupyter_utils import setup_jupyter_notebook
from stocks_dl.constants import TARGET_COLUMN
from stocks_dl.data.pipeline import load_features_multi
from stocks_dl.training.dataset import split_train_test
from stocks_dl.training.train import calculate_metrics

import warnings
warnings.filterwarnings('ignore')


In [3]:
EXPERIMENT_NAME = 'catboost_checkpoint'
setup_jupyter_notebook(environment='prod', experiment=EXPERIMENT_NAME)

# Для локального запуска:
# setup_jupyter_notebook(environment='local', experiment='catboost_test')


2026/06/09 13:15:45 INFO mlflow.tracking.fluent: Experiment with name 'catboost_checkpoint' does not exist. Creating a new experiment.


Environment: prod
APP_CONFIG: config.toml
Tracking URI: http://localhost:5050
S3 endpoint: http://localhost:9050
Experiment: catboost_checkpoint
Database: localhost:15432/stocks_advisor_db


In [4]:
TICKERS = ['SBER', 'TCSG', 'GAZP', 'LKOH', 'ROSN']
SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
ENRICHMENTS_LIMIT = 1_000_000

RUNS_DIR = REPO_ROOT / 'stocks_dl_runs' / 'catboost'
RUNS_DIR.mkdir(parents=True, exist_ok=True)


## Data


In [5]:
features_by_ticker, enrichments_df = load_features_multi(
    TICKERS,
    enrichments_limit=ENRICHMENTS_LIMIT,
)

for ticker, df in features_by_ticker.items():
    print(ticker, df.shape, df['begin'].min(), df['begin'].max())


SBER (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
TCSG (2853, 89) 2022-07-04 10:00:00 2024-11-20 18:00:00
GAZP (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
LKOH (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00
ROSN (5248, 89) 2022-06-02 10:00:00 2026-05-26 18:00:00


## Helpers


In [6]:
def build_catboost_configs(ticker: str) -> list[dict]:
    base = {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': SEED,
        'verbose': False,
        'allow_writing_files': False,
    }
    variants = [
        ('default', {}),
        ('depth4_lr003_iter300', {'depth': 4, 'learning_rate': 0.03, 'iterations': 300, 'l2_leaf_reg': 3}),
        ('depth6_lr003_iter500', {'depth': 6, 'learning_rate': 0.03, 'iterations': 500, 'l2_leaf_reg': 5}),
        ('depth8_lr001_iter700', {'depth': 8, 'learning_rate': 0.01, 'iterations': 700, 'l2_leaf_reg': 7}),
        ('depth6_lr005_iter300', {'depth': 6, 'learning_rate': 0.05, 'iterations': 300, 'l2_leaf_reg': 3}),
    ]
    return [
        {
            'run_name': f'{ticker.lower()}_catboost_{idx:02d}_{name}',
            'ticker': ticker,
            'variant': name,
            **base,
            **params,
        }
        for idx, (name, params) in enumerate(variants, start=1)
    ]


def make_xy(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    feature_cols = [c for c in train_df.columns if c not in ('begin', TARGET_COLUMN)]

    X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_val = val_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

    y_train = train_df[TARGET_COLUMN]
    y_val = val_df[TARGET_COLUMN]
    y_test = test_df[TARGET_COLUMN]

    return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols


def predictions_frame(test_df: pd.DataFrame, y_true, y_pred) -> pd.DataFrame:
    out = pd.DataFrame({
        'begin': pd.to_datetime(test_df['begin']).reset_index(drop=True),
        TARGET_COLUMN: np.asarray(y_true),
        'predict': np.asarray(y_pred),
    })
    out['error'] = out['predict'] - out[TARGET_COLUMN]
    out['abs_error'] = out['error'].abs()
    out['direction_match'] = np.sign(out['predict']) == np.sign(out[TARGET_COLUMN])
    return out


def log_catboost_model(model: CatBoostRegressor, artifact_path: str = 'model') -> None:
    try:
        mlflow.catboost.log_model(model, artifact_path)
    except Exception as exc:
        print(f'mlflow.catboost.log_model failed: {exc}. Saving .cbm artifact instead.')
        with tempfile.TemporaryDirectory() as tmpdir:
            path = Path(tmpdir) / 'model.cbm'
            model.save_model(path)
            mlflow.log_artifact(str(path), artifact_path=artifact_path)


In [7]:
def run_catboost_experiment(
    cfg: dict,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> dict:
    X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = make_xy(train_df, val_df, test_df)

    params = deepcopy(cfg)
    run_name = params.pop('run_name')
    ticker = params.pop('ticker')
    variant = params.pop('variant')

    model = CatBoostRegressor(**params)

    with tempfile.TemporaryDirectory() as tmpdir, mlflow.start_run(run_name=run_name) as run:
        tmpdir = Path(tmpdir)

        mlflow.set_tags({
            'ticker': ticker,
            'stage': 'catboost_search',
            'model_family': 'CatBoost',
            'target': TARGET_COLUMN,
            'variant': variant,
            'seed': str(SEED),
        })
        mlflow.log_params({k: v for k, v in params.items() if isinstance(v, (str, int, float, bool))})
        mlflow.log_param('variant', variant)
        mlflow.log_param('feature_count', len(feature_cols))
        mlflow.log_dict(cfg, 'config.json')

        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        train_metrics = calculate_metrics(y_train.to_numpy(), train_pred)
        val_metrics = calculate_metrics(y_val.to_numpy(), val_pred)
        test_metrics = calculate_metrics(y_test.to_numpy(), test_pred)

        pred_df = predictions_frame(test_df, y_test.to_numpy(), test_pred)
        pred_df.to_csv(tmpdir / 'test_predictions.csv', index=False)

        importance_df = pd.DataFrame({
            'feature': feature_cols,
            'importance': model.get_feature_importance(),
        }).sort_values('importance', ascending=False)
        importance_df.to_csv(tmpdir / 'feature_importance.csv', index=False)

        mlflow.log_artifacts(str(tmpdir))
        log_catboost_model(model)

        summary = {
            'run_id': run.info.run_id,
            'run_name': run_name,
            'ticker': ticker,
            'variant': variant,
            'iterations': cfg.get('iterations', None),
            'depth': cfg.get('depth', None),
            'learning_rate': cfg.get('learning_rate', None),
            'l2_leaf_reg': cfg.get('l2_leaf_reg', None),
            'feature_count': len(feature_cols),
            'best_iteration': model.get_best_iteration(),
            'train_mae': float(train_metrics['mae']),
            'train_rmse': float(train_metrics['rmse']),
            'train_r2': float(train_metrics['r2']),
            'train_direction_accuracy': float(train_metrics['direction_accuracy']),
            'val_mae': float(val_metrics['mae']),
            'val_rmse': float(val_metrics['rmse']),
            'val_r2': float(val_metrics['r2']),
            'val_direction_accuracy': float(val_metrics['direction_accuracy']),
            'test_mae': float(test_metrics['mae']),
            'test_rmse': float(test_metrics['rmse']),
            'test_r2': float(test_metrics['r2']),
            'test_direction_accuracy': float(test_metrics['direction_accuracy']),
            'config_json': json.dumps(cfg, ensure_ascii=False),
        }
        mlflow.log_metrics({k: v for k, v in summary.items() if isinstance(v, (int, float, np.floating))})

    print(f'[{run_name}] val_dir_acc={summary["val_direction_accuracy"]:.4f} test_dir_acc={summary["test_direction_accuracy"]:.4f}')
    return summary


## Experiments


In [8]:
mlflow.set_experiment(EXPERIMENT_NAME)

all_rows = []
ticker_summaries = {}

for ticker in TICKERS:
    print(f'\n=== {ticker} ===')
    features_df = features_by_ticker[ticker].copy()
    train_df, val_df, test_df = split_train_test(
        features_df,
        target_column=TARGET_COLUMN,
        test_size=TEST_SIZE,
        val_size=VAL_SIZE,
    )

    rows = []
    for cfg in build_catboost_configs(ticker):
        rows.append(run_catboost_experiment(cfg, train_df, val_df, test_df))

    summary_df = (
        pd.DataFrame(rows)
        .sort_values(['val_direction_accuracy', 'val_r2', 'val_rmse'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    ticker_summaries[ticker] = summary_df
    all_rows.extend(rows)

    out_path = RUNS_DIR / f'{ticker.lower()}_catboost_summary.csv'
    summary_df.to_csv(out_path, index=False)
    display(summary_df)



=== SBER ===


2026/06/09 13:19:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:19:38 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:19:38 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:19:38 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements via 'uv export'.
2026/06/09 13:19:39 INFO mlflow.utils.uv_utils: Exported 227 dependencies via uv
2026/06/09 13:19:39 INFO mlflow.utils.environment: Successfully exported 227 requirements from uv project. Skipping package capture based inference.
2026/06/09 13:19:44 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environ

🏃 View run sber_catboost_01_default at: http://localhost:5050/#/experiments/5/runs/af19b7ae55e844b9917ae7a2710fba70
🧪 View experiment at: http://localhost:5050/#/experiments/5
[sber_catboost_01_default] val_dir_acc=0.5214 test_dir_acc=0.5457


2026/06/09 13:19:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:19:46 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:19:46 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:19:46 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements via 'uv export'.
2026/06/09 13:19:46 INFO mlflow.utils.uv_utils: Exported 227 dependencies via uv
2026/06/09 13:19:46 INFO mlflow.utils.environment: Successfully exported 227 requirements from uv project. Skipping package capture based inference.
2026/06/09 13:19:47 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environ

🏃 View run sber_catboost_02_depth4_lr003_iter300 at: http://localhost:5050/#/experiments/5/runs/c831455607fc4b6eadded545a30ddc11
🧪 View experiment at: http://localhost:5050/#/experiments/5
[sber_catboost_02_depth4_lr003_iter300] val_dir_acc=0.6095 test_dir_acc=0.5676


...

2026/06/09 13:22:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:22:37 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:22:37 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:22:37 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements via 'uv export'.
2026/06/09 13:22:37 INFO mlflow.utils.uv_utils: Exported 227 dependencies via uv
2026/06/09 13:22:37 INFO mlflow.utils.environment: Successfully exported 227 requirements from uv project. Skipping package capture based inference.
2026/06/09 13:22:37 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environ

🏃 View run rosn_catboost_04_depth8_lr001_iter700 at: http://localhost:5050/#/experiments/5/runs/a0e129431e4846fa886e2d6943c97f14
🧪 View experiment at: http://localhost:5050/#/experiments/5
[rosn_catboost_04_depth8_lr001_iter700] val_dir_acc=0.4488 test_dir_acc=0.4133


2026/06/09 13:22:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:22:41 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:22:41 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor
2026/06/09 13:22:41 INFO mlflow.utils.environment: Detected uv project at C:\Users\baben_bakg1j1\HSE\annual_project\stocks-advisor. Attempting to export requirements via 'uv export'.
2026/06/09 13:22:41 INFO mlflow.utils.uv_utils: Exported 227 dependencies via uv
2026/06/09 13:22:41 INFO mlflow.utils.environment: Successfully exported 227 requirements from uv project. Skipping package capture based inference.
2026/06/09 13:22:41 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environ

🏃 View run rosn_catboost_05_depth6_lr005_iter300 at: http://localhost:5050/#/experiments/5/runs/b6b18f03278a4fdb8b8c598fcc304816
🧪 View experiment at: http://localhost:5050/#/experiments/5
[rosn_catboost_05_depth6_lr005_iter300] val_dir_acc=0.4488 test_dir_acc=0.4133


,run_id,run_name,ticker,variant,iterations,depth,learning_rate,l2_leaf_reg,feature_count,best_iteration,...,train_direction_accuracy,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy,config_json
0,d596cf7cad3e46329a710297d9f488f1,rosn_catboost_01_default,ROSN,default,NaN,NaN,NaN,NaN,87,0,...,0.574747,3.722368,4.630430,-0.066273,0.448810,3.036764,4.124029,-0.044579,0.413333,"{""run_name"": ""rosn_catboost_01_default"", ""tick..."
1,b6b18f03278a4fdb8b8c598fcc304816,rosn_catboost_05_depth6_lr005_iter300,ROSN,depth6_lr005_iter300,300.0,6.0,0.05,3.0,87,0,...,0.574747,3.725443,4.633013,-0.067463,0.448810,3.034249,4.121640,-0.043369,0.413333,"{""run_name"": ""rosn_catboost_05_depth6_lr005_it..."
2,e5ac03760e6b463bb6caab2c782b01df,rosn_catboost_03_depth6_lr003_iter500,ROSN,depth6_lr003_iter500,500.0,6.0,0.03,5.0,87,0,...,0.545861,3.730885,4.637706,-0.069626,0.448810,3.029926,4.117486,-0.041267,0.413333,"{""run_name"": ""rosn_catboost_03_depth6_lr003_it..."
3,a0e129431e4846fa886e2d6943c97f14,rosn_catboost_04_depth8_lr001_iter700,ROSN,depth8_lr001_iter700,700.0,8.0,0.01,7.0,87,0,...,0.545861,3.748461,4.657674,-0.078857,0.448810,3.033315,4.124349,-0.044742,0.413333,"{""run_name"": ""rosn_catboost_04_depth8_lr001_it..."
4,b0f4ee50884f4f9a81be67a6834c9d4d,rosn_catboost_02_depth4_lr003_iter300,ROSN,depth4_lr003_iter300,300.0,4.0,0.03,3.0,87,0,...,0.549136,3.732099,4.639823,-0.070603,0.447619,3.025437,4.114479,-0.039747,0.413333,"{""run_name"": ""rosn_catboost_02_depth4_lr003_it..."


## Best models


In [9]:
catboost_runs_summary_df = (
    pd.DataFrame(all_rows)
    .sort_values(['ticker', 'val_direction_accuracy', 'val_r2', 'val_rmse'], ascending=[True, False, False, True])
    .reset_index(drop=True)
)
catboost_runs_summary_df.to_csv(RUNS_DIR / 'catboost_runs_summary.csv', index=False)
catboost_runs_summary_df


,run_id,run_name,ticker,variant,iterations,depth,learning_rate,l2_leaf_reg,feature_count,best_iteration,...,train_direction_accuracy,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy,config_json
0,8f7e67f7aa7b48f4a63ab6f95f41a276,gazp_catboost_03_depth6_lr003_iter500,GAZP,depth6_lr003_iter500,500.0,6.0,0.03,5.0,87,192,...,0.799285,4.588292,5.866433,0.048223,0.661905,3.040585,4.146352,-0.970949,0.591429,"{""run_name"": ""gazp_catboost_03_depth6_lr003_it..."
1,12cc4d424a6e408f9cbe1f46aee6f086,gazp_catboost_04_depth8_lr001_iter700,GAZP,depth8_lr001_iter700,700.0,8.0,0.01,7.0,87,483,...,0.820429,4.621567,5.953455,0.019777,0.640476,2.862275,3.809716,-0.663904,0.542857,"{""run_name"": ""gazp_catboost_04_depth8_lr001_it..."
2,aed81301a17f4a6a88cee8b9a86bb850,gazp_catboost_02_depth4_lr003_iter300,GAZP,depth4_lr003_iter300,300.0,4.0,0.03,3.0,87,296,...,0.801072,4.509632,5.684408,0.106371,0.636905,3.122039,4.258419,-1.078929,0.620000,"{""run_name"": ""gazp_catboost_02_depth4_lr003_it..."
3,808126db7a104b33857d0ba7cf20a8cc,gazp_catboost_01_default,GAZP,default,NaN,NaN,NaN,NaN,87,192,...,0.896962,4.585702,5.950694,0.020686,0.614286,3.195167,4.353868,-1.173169,0.576190,"{""run_name"": ""gazp_catboost_01_default"", ""tick..."
4,e82de64e20bc44e4b514d9dc5fcab3c5,gazp_catboost_05_depth6_lr005_iter300,GAZP,depth6_lr005_iter300,300.0,6.0,0.05,3.0,87,3,...,0.606611,4.817542,6.058257,-0.015038,0.585714,2.348969,2.933129,0.013709,0.594286,"{""run_name"": ""gazp_catboost_05_depth6_lr005_it..."
5,217f120eb71c4ef7b5a3c68df4fdf353,lkoh_catboost_01_default,LKOH,default,NaN,NaN,NaN,NaN,87,31,...,0.780226,2.868443,3.628466,-0.036298,0.548810,3.164152,4.222204,-0.142079,0.471429,"{""run_name"": ""lkoh_catboost_01_default"", ""tick..."
6,a6201fe6cfc2485ca8a053d97f60acba,lkoh_catboost_03_depth6_lr003_iter500,LKOH,depth6_lr003_iter500,500.0,6.0,0.03,5.0,87,51,...,0.763252,2.863236,3.637031,-0.041196,0.509524,3.142826,4.198248,-0.129156,0.472381,"{""run_name"": ""lkoh_catboost_03_depth6_lr003_it..."
7,f10dfc0bc4af430294bbd6ed07b83314,lkoh_catboost_02_depth4_lr003_iter300,LKOH,depth4_lr003_iter300,300.0,4.0,0.03,3.0,87,70,...,0.748958,2.948247,3.669277,-0.059740,0.507143,3.325122,4.336116,-0.204535,0.370476,"{""run_name"": ""lkoh_catboost_02_depth4_lr003_it..."
8,2d1cb916aa844027a01214126c4ca57a,lkoh_catboost_05_depth6_lr005_iter300,LKOH,depth6_lr005_iter300,300.0,6.0,0.05,3.0,87,15,...,0.686420,2.900617,3.666255,-0.057995,0.492857,3.172916,4.190699,-0.125099,0.371429,"{""run_name"": ""lkoh_catboost_05_depth6_lr005_it..."
9,b861b027d8c947eabfcb41392be658e9,lkoh_catboost_04_depth8_lr001_iter700,LKOH,depth8_lr001_iter700,700.0,8.0,0.01,7.0,87,69,...,0.717094,2.949458,3.716798,-0.087367,0.482143,3.147494,4.191918,-0.125753,0.391429,"{""run_name"": ""lkoh_catboost_04_depth8_lr001_it..."


In [10]:
best_models_rows = []

for ticker, summary_df in ticker_summaries.items():
    best_row = summary_df.sort_values(
        ['val_direction_accuracy', 'val_r2', 'val_rmse'],
        ascending=[False, False, True],
    ).iloc[0]
    best_models_rows.append({
        'ticker': ticker,
        'run_name': best_row['run_name'],
        'run_id': best_row['run_id'],
        'variant': best_row['variant'],
        'iterations': best_row['iterations'],
        'depth': best_row['depth'],
        'learning_rate': best_row['learning_rate'],
        'l2_leaf_reg': best_row['l2_leaf_reg'],
        'best_iteration': best_row['best_iteration'],
        'feature_count': best_row['feature_count'],
        'val_mae': best_row['val_mae'],
        'val_rmse': best_row['val_rmse'],
        'val_r2': best_row['val_r2'],
        'val_direction_accuracy': best_row['val_direction_accuracy'],
        'test_mae': best_row['test_mae'],
        'test_rmse': best_row['test_rmse'],
        'test_r2': best_row['test_r2'],
        'test_direction_accuracy': best_row['test_direction_accuracy'],
    })

best_models_summary_df = pd.DataFrame(best_models_rows).sort_values('ticker').reset_index(drop=True)
best_models_summary_df.to_csv(RUNS_DIR / 'catboost_best_models_summary.csv', index=False)
best_models_summary_df


,ticker,run_name,run_id,variant,iterations,depth,learning_rate,l2_leaf_reg,best_iteration,feature_count,val_mae,val_rmse,val_r2,val_direction_accuracy,test_mae,test_rmse,test_r2,test_direction_accuracy
0,GAZP,gazp_catboost_03_depth6_lr003_iter500,8f7e67f7aa7b48f4a63ab6f95f41a276,depth6_lr003_iter500,500.0,6.0,0.03,5.0,192,87,4.588292,5.866433,0.048223,0.661905,3.040585,4.146352,-0.970949,0.591429
1,LKOH,lkoh_catboost_01_default,217f120eb71c4ef7b5a3c68df4fdf353,default,NaN,NaN,NaN,NaN,31,87,2.868443,3.628466,-0.036298,0.548810,3.164152,4.222204,-0.142079,0.471429
2,ROSN,rosn_catboost_01_default,d596cf7cad3e46329a710297d9f488f1,default,NaN,NaN,NaN,NaN,0,87,3.722368,4.630430,-0.066273,0.448810,3.036764,4.124029,-0.044579,0.413333
3,SBER,sber_catboost_02_depth4_lr003_iter300,c831455607fc4b6eadded545a30ddc11,depth4_lr003_iter300,300.0,4.0,0.03,3.0,100,87,2.196853,2.807435,-0.012685,0.609524,1.771416,2.275850,-0.488548,0.567619
4,TCSG,tcsg_catboost_03_depth6_lr003_iter500,a8b9477d55fd4958800ad0a67f7faf98,depth6_lr003_iter500,500.0,6.0,0.03,5.0,317,87,2.220265,2.762466,0.122631,0.623632,4.141017,5.303353,-0.202564,0.423818
